**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Sessions 37-38: 프로젝트 배포 및 어플리케이션 구현

## 🎯 모델 배포 및 어플리케이션 구현

이 노트북에서는 파인튜닝된 모델을 **실제 서비스로 배포**하고, **웹 어플리케이션**을 구현합니다.

### 배포 파이프라인
```
모델 양자화(GGUF) → Ollama 등록 → FastAPI 서버 → Streamlit 웹앱 → 데모 준비
```

### 사전 준비
- 파인튜닝된 모델 (merged_model 또는 final_model)
- `final_report.json` 평가 보고서
- Ollama 설치 (`curl -fsSL https://ollama.com/install.sh | sh`)
- llama.cpp 도구 (GGUF 변환용)

In [ ]:
# =====================================================
# 📦 필수 패키지 확인
# =====================================================

# !pip install fastapi uvicorn streamlit requests

import json
import os
import subprocess

PROJECT_DIR = "./my_project"  # TODO: 프로젝트 디렉토리

# 프로젝트 정보 로드
with open(f"{PROJECT_DIR}/project_plan.json", "r", encoding="utf-8") as f:
    project_plan = json.load(f)

MODEL_NAME = project_plan["model_config"]["base_model"]
MERGED_MODEL_PATH = f"{PROJECT_DIR}/models/merged_model"  # TODO: 병합 모델 경로

print(f"📋 프로젝트: {project_plan['project_name']}")
print(f"🤖 모델: {MODEL_NAME}")
print(f"📂 모델 경로: {MERGED_MODEL_PATH}")

# Ollama 설치 확인
try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"\n✅ Ollama: {result.stdout.strip()}")
except FileNotFoundError:
    print("⚠️ Ollama가 설치되어 있지 않습니다. 아래 명령어로 설치하세요:")
    print("   curl -fsSL https://ollama.com/install.sh | sh")

---
## 1️⃣ 모델 양자화 (GGUF 변환)

파인튜닝된 모델을 **GGUF 형식**으로 변환합니다. GGUF는 Ollama, llama.cpp 등에서 사용하는 경량 모델 형식입니다.

### 양자화 옵션
| 양자화 타입 | 모델 크기 (1.5B) | 품질 | 속도 |
|------------|-----------------|------|------|
| Q8_0 | ~1.6GB | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| Q5_K_M | ~1.1GB | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Q4_K_M | ~0.9GB | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Q4_0 | ~0.8GB | ⭐⭐ | ⭐⭐⭐⭐⭐ |

In [ ]:
# =====================================================
# 🔧 GGUF 변환 (llama.cpp 사용)
# TODO: llama.cpp 경로와 양자화 타입을 설정하세요
# =====================================================

# llama.cpp 경로 (사전 설치 필요)
# git clone https://github.com/ggerganov/llama.cpp
# cd llama.cpp && pip install -r requirements.txt
LLAMA_CPP_DIR = ""  # TODO: llama.cpp 디렉토리 경로

# 양자화 설정
QUANT_TYPE = "Q4_K_M"  # TODO: 양자화 타입 선택 (Q4_K_M, Q5_K_M, Q8_0)
GGUF_OUTPUT_DIR = f"{PROJECT_DIR}/models/gguf"
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

if LLAMA_CPP_DIR and os.path.exists(LLAMA_CPP_DIR):
    # Step 1: HuggingFace → GGUF 변환
    print("🔄 Step 1: HuggingFace → GGUF 변환")
    gguf_fp16_path = f"{GGUF_OUTPUT_DIR}/model-fp16.gguf"
    
    convert_cmd = f"""python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py \
        {MERGED_MODEL_PATH} \
        --outfile {gguf_fp16_path} \
        --outtype f16"""
    
    print(f"   명령어: {convert_cmd}")
    # TODO: 아래 주석을 해제하여 실행하세요
    # os.system(convert_cmd)
    
    # Step 2: 양자화
    print(f"\n🔄 Step 2: {QUANT_TYPE} 양자화")
    gguf_quant_path = f"{GGUF_OUTPUT_DIR}/model-{QUANT_TYPE}.gguf"
    
    quant_cmd = f"""{LLAMA_CPP_DIR}/llama-quantize \
        {gguf_fp16_path} \
        {gguf_quant_path} \
        {QUANT_TYPE}"""
    
    print(f"   명령어: {quant_cmd}")
    # TODO: 아래 주석을 해제하여 실행하세요
    # os.system(quant_cmd)
    
    print(f"\n✅ GGUF 파일: {gguf_quant_path}")
else:
    print("⚠️ LLAMA_CPP_DIR을 설정해주세요.")
    print("\n📌 llama.cpp 설치 방법:")
    print("   git clone https://github.com/ggerganov/llama.cpp")
    print("   cd llama.cpp")
    print("   pip install -r requirements.txt")
    print("   make  # 또는 cmake로 빌드")

---
## 2️⃣ Ollama 모델 등록

GGUF 모델을 **Ollama에 등록**하여 간편하게 서빙합니다.

In [ ]:
# =====================================================
# 📝 Ollama Modelfile 생성
# TODO: 시스템 프롬프트와 파라미터를 수정하세요
# =====================================================

# TODO: 프로젝트에 맞게 수정하세요
OLLAMA_MODEL_NAME = "my-project-model"  # TODO: Ollama에 등록할 모델 이름
GGUF_PATH = f"{GGUF_OUTPUT_DIR}/model-{QUANT_TYPE}.gguf"  # GGUF 파일 경로

# TODO: 시스템 프롬프트를 도메인에 맞게 작성하세요
SYSTEM_PROMPT = """당신은 도움이 되는 AI 어시스턴트입니다.
사용자의 질문에 정확하고 친절하게 답변해주세요."""  # TODO: 수정하세요

# Modelfile 생성
modelfile_content = f"""FROM {os.path.abspath(GGUF_PATH)}

SYSTEM \"\"\"
{SYSTEM_PROMPT}
\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
"""

modelfile_path = f"{PROJECT_DIR}/Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print(f"✅ Modelfile 생성: {modelfile_path}")
print(f"\n📝 내용:")
print(modelfile_content)

In [ ]:
# =====================================================
# 🚀 Ollama 모델 등록 및 테스트
# TODO: 아래 명령어를 실행하세요
# =====================================================

print("📌 터미널에서 아래 명령어를 실행하세요:")
print(f"\n1. 모델 등록:")
print(f"   ollama create {OLLAMA_MODEL_NAME} -f {os.path.abspath(modelfile_path)}")
print(f"\n2. 모델 테스트:")
print(f"   ollama run {OLLAMA_MODEL_NAME}")
print(f"\n3. 모델 목록 확인:")
print(f"   ollama list")

# 프로그래밍으로 실행 (선택사항)
# TODO: 아래 주석을 해제하여 실행하세요
# os.system(f"ollama create {OLLAMA_MODEL_NAME} -f {os.path.abspath(modelfile_path)}")

In [ ]:
# =====================================================
# 🧪 Ollama API 테스트
# TODO: Ollama가 실행 중인지 확인 후 테스트하세요
# =====================================================

import requests

OLLAMA_URL = "http://localhost:11434"  # Ollama 기본 URL

def test_ollama(model_name, prompt):
    """Ollama API로 모델을 테스트합니다."""
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": model_name,
                "prompt": prompt,
                "stream": False,
            },
            timeout=60,
        )
        if response.status_code == 200:
            return response.json()["response"]
        else:
            return f"Error: {response.status_code}"
    except Exception as e:
        return f"Error: {e}"

# TODO: 테스트 프롬프트 작성
test_prompt = ""  # TODO: 도메인에 맞는 테스트 질문

if test_prompt:
    print(f"❓ 질문: {test_prompt}")
    response = test_ollama(OLLAMA_MODEL_NAME, test_prompt)
    print(f"🤖 응답: {response}")
else:
    print("⚠️ test_prompt를 작성해주세요.")

---
## 3️⃣ API 서버 구현 (FastAPI)

FastAPI를 사용하여 **REST API 서버**를 구현합니다.

> 💡 **Hint**: FastAPI 서버는 별도의 Python 파일로 저장하여 터미널에서 실행합니다.

In [ ]:
# =====================================================
# 🌐 FastAPI 서버 코드 생성
# TODO: 엔드포인트와 로직을 커스터마이징하세요
# =====================================================

fastapi_code = '''#!/usr/bin/env python3
"""
FastAPI 서버 - {project_name}
실행: uvicorn api_server:app --host 0.0.0.0 --port 8000 --reload
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import requests
import json

app = FastAPI(
    title="{project_name} API",
    description="도메인 특화 sLLM API 서버",
    version="1.0.0",
)

# ============================
# 설정
# ============================
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "{ollama_model_name}"


# ============================
# 요청/응답 모델
# ============================
class ChatRequest(BaseModel):
    message: str
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 512


class ChatResponse(BaseModel):
    response: str
    model: str


# ============================
# 엔드포인트
# ============================
@app.get("/")
async def root():
    return {{"message": "{project_name} API", "status": "running"}}


@app.get("/health")
async def health_check():
    """서버 상태 확인"""
    try:
        response = requests.get(f"{{OLLAMA_URL}}/api/tags", timeout=5)
        return {{"status": "healthy", "ollama": "connected"}}
    except Exception:
        return {{"status": "degraded", "ollama": "disconnected"}}


@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """채팅 엔드포인트"""
    try:
        response = requests.post(
            f"{{OLLAMA_URL}}/api/generate",
            json={{
                "model": MODEL_NAME,
                "prompt": request.message,
                "stream": False,
                "options": {{
                    "temperature": request.temperature,
                    "num_predict": request.max_tokens,
                }},
            }},
            timeout=120,
        )
        
        if response.status_code == 200:
            result = response.json()
            return ChatResponse(
                response=result["response"],
                model=MODEL_NAME,
            )
        else:
            raise HTTPException(status_code=500, detail="Ollama 응답 오류")
    except requests.exceptions.ConnectionError:
        raise HTTPException(status_code=503, detail="Ollama 서버에 연결할 수 없습니다.")


# TODO: 도메인에 맞는 추가 엔드포인트를 구현하세요
# 예시: 특정 형식의 응답을 반환하는 엔드포인트
# @app.post("/analyze")
# async def analyze(request: ChatRequest):
#     """도메인 특화 분석 엔드포인트"""
#     # TODO: 구현
#     pass


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''.format(
    project_name=project_plan["project_name"],
    ollama_model_name=OLLAMA_MODEL_NAME,
)

# 파일 저장
api_path = f"{PROJECT_DIR}/app/api_server.py"
os.makedirs(f"{PROJECT_DIR}/app", exist_ok=True)
with open(api_path, "w", encoding="utf-8") as f:
    f.write(fastapi_code)

print(f"✅ FastAPI 서버 코드 생성: {api_path}")
print(f"\n📌 실행 방법:")
print(f"   cd {os.path.abspath(PROJECT_DIR)}/app")
print(f"   uvicorn api_server:app --host 0.0.0.0 --port 8000 --reload")
print(f"\n📌 API 문서: http://localhost:8000/docs")

---
## 4️⃣ Streamlit 웹앱 구현

**Streamlit**을 사용하여 데모용 웹 어플리케이션을 구현합니다.

> 💡 **Hint**: Streamlit 앱은 별도의 Python 파일로 저장하여 `streamlit run` 명령어로 실행합니다.

In [ ]:
# =====================================================
# 🎨 Streamlit 웹앱 코드 생성
# TODO: UI와 기능을 커스터마이징하세요
# =====================================================

streamlit_code = '''#!/usr/bin/env python3
"""
Streamlit 웹앱 - {project_name}
실행: streamlit run streamlit_app.py --server.port 8501
"""

import streamlit as st
import requests
import json

# ============================
# 페이지 설정
# ============================
st.set_page_config(
    page_title="{project_name}",
    page_icon="🤖",
    layout="wide",
)

# ============================
# 설정
# ============================
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "{ollama_model_name}"


# ============================
# Ollama API 호출 함수
# ============================
def get_response(prompt, temperature=0.7, max_tokens=512):
    """Ollama API를 호출하여 응답을 받습니다."""
    try:
        response = requests.post(
            f"{{OLLAMA_URL}}/api/generate",
            json={{
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "options": {{
                    "temperature": temperature,
                    "num_predict": max_tokens,
                }},
            }},
            timeout=120,
        )
        if response.status_code == 200:
            return response.json()["response"]
        else:
            return f"오류 발생: {{response.status_code}}"
    except Exception as e:
        return f"연결 오류: {{e}}"


# ============================
# 사이드바
# ============================
with st.sidebar:
    st.title("⚙️ 설정")
    
    temperature = st.slider("Temperature", 0.0, 1.0, 0.7, 0.1)
    max_tokens = st.slider("최대 토큰 수", 64, 1024, 512, 64)
    
    st.divider()
    
    st.markdown("### 📌 모델 정보")
    st.write(f"모델: {{MODEL_NAME}}")
    
    # 서버 상태 확인
    try:
        r = requests.get(f"{{OLLAMA_URL}}/api/tags", timeout=3)
        st.success("Ollama 연결됨")
    except Exception:
        st.error("Ollama 연결 실패")
    
    st.divider()
    
    # TODO: 도메인에 맞는 예시 질문 추가
    st.markdown("### 💡 예시 질문")
    example_questions = [
        "",  # TODO: 예시 질문 1
        "",  # TODO: 예시 질문 2
        "",  # TODO: 예시 질문 3
    ]
    for q in example_questions:
        if q and st.button(q[:30] + "...", key=q):
            st.session_state["input_text"] = q


# ============================
# 메인 영역
# ============================
st.title("🤖 {project_name}")
st.markdown("---")

# TODO: 프로젝트 설명을 작성하세요
st.markdown("""
> 이 어플리케이션은 도메인 특화 파인튜닝된 sLLM을 활용합니다.
> 아래에 질문을 입력하면 AI가 답변을 생성합니다.
""")

# 채팅 히스토리 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 채팅 히스토리 표시
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 사용자 입력
if prompt := st.chat_input("질문을 입력하세요..."):
    # 사용자 메시지 표시
    st.session_state.messages.append({{"role": "user", "content": prompt}})
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # AI 응답 생성
    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            response = get_response(prompt, temperature, max_tokens)
        st.markdown(response)
    
    # 히스토리에 추가
    st.session_state.messages.append({{"role": "assistant", "content": response}})

# 채팅 초기화 버튼
if st.button("🗑️ 대화 초기화"):
    st.session_state.messages = []
    st.rerun()
'''.format(
    project_name=project_plan["project_name"],
    ollama_model_name=OLLAMA_MODEL_NAME,
)

# 파일 저장
streamlit_path = f"{PROJECT_DIR}/app/streamlit_app.py"
with open(streamlit_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print(f"✅ Streamlit 앱 코드 생성: {streamlit_path}")
print(f"\n📌 실행 방법:")
print(f"   streamlit run {os.path.abspath(streamlit_path)} --server.port 8501")
print(f"\n📌 접속: http://localhost:8501")

---
## 5️⃣ 데모 시나리오 준비

최종 발표에서 사용할 **데모 시나리오**를 준비합니다.

In [ ]:
# =====================================================
# 🎬 데모 시나리오 템플릿
# TODO: 데모 시나리오를 작성하세요
# =====================================================

demo_scenarios = [
    {
        "title": "",           # TODO: 시나리오 제목 (예: "기본 질의응답")
        "description": "",     # TODO: 시나리오 설명
        "prompts": [
            "",                 # TODO: 데모 프롬프트 1
            "",                 # TODO: 데모 프롬프트 2
        ],
        "expected_behavior": "",  # TODO: 기대되는 동작
    },
    {
        "title": "",           # TODO: 시나리오 2 제목
        "description": "",
        "prompts": [
            "",
            "",
        ],
        "expected_behavior": "",
    },
    {
        "title": "",           # TODO: 시나리오 3 제목 (엣지 케이스/한계)
        "description": "",
        "prompts": [
            "",
        ],
        "expected_behavior": "",
    },
]

# 시나리오 출력
for i, scenario in enumerate(demo_scenarios, 1):
    if scenario["title"]:
        print(f"\n🎬 시나리오 {i}: {scenario['title']}")
        print(f"   설명: {scenario['description']}")
        for j, p in enumerate(scenario['prompts'], 1):
            if p:
                print(f"   질문 {j}: {p}")
        print(f"   기대 동작: {scenario['expected_behavior']}")

# 시나리오 저장
with open(f"{PROJECT_DIR}/app/demo_scenarios.json", "w", encoding="utf-8") as f:
    json.dump(demo_scenarios, f, ensure_ascii=False, indent=2)

print(f"\n💾 데모 시나리오 저장 완료")

---
## 6️⃣ 최종 발표 자료 체크리스트

최종 발표에 포함해야 할 항목을 체크합니다.

In [ ]:
# =====================================================
# ✅ 최종 발표 체크리스트
# TODO: 각 항목을 확인하고 True로 변경하세요
# =====================================================

presentation_checklist = {
    "1. 프로젝트 개요": {
        "프로젝트 이름 및 도메인": False,      # TODO
        "문제 정의 및 목표": False,
        "대상 사용자": False,
    },
    "2. 데이터": {
        "데이터 수집 방법": False,
        "데이터 통계 (수량, 형식)": False,
        "데이터 정제/증강 과정": False,
    },
    "3. 모델 학습": {
        "베이스 모델 선택 근거": False,
        "학습 방법론 (LoRA/QLoRA/FFT)": False,
        "하이퍼파라미터 설정": False,
        "학습 로그 (Loss 그래프)": False,
    },
    "4. 평가 결과": {
        "자동 메트릭 (ROUGE, BLEU)": False,
        "LLM-as-a-Judge 결과": False,
        "Before/After 비교": False,
        "개선 과정 설명": False,
    },
    "5. 배포": {
        "양자화 방법": False,
        "API 서버 아키텍처": False,
        "웹앱 데모": False,
    },
    "6. 결론": {
        "주요 성과": False,
        "한계점 및 향후 과제": False,
        "배운 점": False,
    },
}

# 체크리스트 출력
total = 0
done = 0
for section, items in presentation_checklist.items():
    section_done = sum(1 for v in items.values() if v)
    section_total = len(items)
    total += section_total
    done += section_done
    status = "✅" if section_done == section_total else "🔲"
    print(f"\n{status} {section} ({section_done}/{section_total})")
    for item, completed in items.items():
        mark = "✅" if completed else "⬜"
        print(f"   {mark} {item}")

print(f"\n{'='*40}")
print(f"전체 완성도: {done}/{total} ({done/total*100:.0f}%)")
progress = int(done/total * 20)
print(f"[{'█' * progress}{'░' * (20-progress)}]")

---
## 7️⃣ 프로젝트 회고

프로젝트 전체를 돌아보며 **회고(Retrospective)**를 작성합니다.

In [ ]:
# =====================================================
# 📝 프로젝트 회고
# TODO: 각 항목을 채워주세요
# =====================================================

retrospective = {
    # 잘한 점 (Keep)
    "keep": [
        "",  # TODO: 잘한 점 1
        "",  # TODO: 잘한 점 2
        "",  # TODO: 잘한 점 3
    ],
    
    # 문제점 (Problem)
    "problem": [
        "",  # TODO: 문제점 1
        "",  # TODO: 문제점 2
    ],
    
    # 시도해볼 것 (Try)
    "try": [
        "",  # TODO: 다음에 시도할 것 1
        "",  # TODO: 다음에 시도할 것 2
    ],
    
    # 배운 점
    "learned": [
        "",  # TODO: 배운 점 1
        "",  # TODO: 배운 점 2
        "",  # TODO: 배운 점 3
    ],
    
    # 핵심 교훈
    "key_takeaway": "",  # TODO: 이 프로젝트의 핵심 교훈 한 줄
}

# 회고 출력
print("📝 프로젝트 회고")
print("="*50)

print("\n🟢 Keep (잘한 점):")
for item in retrospective["keep"]:
    if item:
        print(f"   + {item}")

print("\n🔴 Problem (문제점):")
for item in retrospective["problem"]:
    if item:
        print(f"   - {item}")

print("\n🟡 Try (다음에 시도):")
for item in retrospective["try"]:
    if item:
        print(f"   → {item}")

print("\n📚 Learned (배운 점):")
for item in retrospective["learned"]:
    if item:
        print(f"   📌 {item}")

if retrospective["key_takeaway"]:
    print(f"\n💎 핵심 교훈: {retrospective['key_takeaway']}")

# 회고 저장
with open(f"{PROJECT_DIR}/results/retrospective.json", "w", encoding="utf-8") as f:
    json.dump(retrospective, f, ensure_ascii=False, indent=2)
print(f"\n💾 회고 저장 완료")

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 모델 양자화 (GGUF 변환)
- ✅ Ollama 모델 등록 및 테스트
- ✅ FastAPI REST API 서버 구현
- ✅ Streamlit 웹앱 구현
- ✅ 데모 시나리오 준비
- ✅ 최종 발표 체크리스트 확인
- ✅ 프로젝트 회고 작성

### 산출물
- 📁 `my_project/models/gguf/` - GGUF 양자화 모델
- 📁 `my_project/Modelfile` - Ollama 모델 설정
- 📁 `my_project/app/api_server.py` - FastAPI 서버
- 📁 `my_project/app/streamlit_app.py` - Streamlit 웹앱
- 📁 `my_project/app/demo_scenarios.json` - 데모 시나리오
- 📁 `my_project/results/retrospective.json` - 프로젝트 회고

### 실행 순서 (배포 시)
```bash
# 1. Ollama 서버 시작
ollama serve

# 2. 모델 등록
ollama create my-project-model -f Modelfile

# 3. (선택) FastAPI 서버 시작
cd my_project/app
uvicorn api_server:app --host 0.0.0.0 --port 8000

# 4. Streamlit 앱 시작
streamlit run streamlit_app.py --server.port 8501
```

### 프로젝트 전체 완료!
🎉 **축하합니다!** 도메인 특화 sLLM 파인튜닝 프로젝트를 완료했습니다.

여러분은 이제 다음을 수행할 수 있습니다:
1. 도메인에 맞는 데이터 파이프라인 구축
2. LoRA/QLoRA 기반 효율적인 파인튜닝
3. 다각도 모델 평가 및 반복 개선
4. 양자화 및 실서비스 배포